# ME 313 · Lab 2.1 — PINN the Bus-Bar (Module 2)  ⭐ *showpiece*
**Goal:** solve the heat-generation problem with a **physics-informed neural network (PINN)** and trust it *only* because it reproduces the analytical answer.

$$\frac{d^2T}{dx^2}+\frac{\dot g}{k}=0,\qquad \left.\frac{dT}{dx}\right|_{0}=0,\qquad -k\left.\frac{dT}{dx}\right|_{L}=h\,[T(L)-T_\infty]$$

with $\dot g=2\times10^5$, $k=50$, $L=0.05$, $h=100$, $T_\infty=300$ → the exact solution is $T(x)=405-2000x^2$.

> **Runs on free Google Colab** (PyTorch is pre-installed). The two reference cells below (analytical + finite difference) are verified; **run the PINN cell once before presenting** to confirm convergence on your machine.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
g,k,L,h,Tinf = 2e5, 50., 0.05, 100., 300.
T_analytic = lambda x: 405 - 2000*x**2          # exact solution to beat


### 1. A verified reference — finite differences (2nd-order BCs)
Before trusting any neural net, build a reference we *can* check. This reproduces the analytical curve to machine precision.


In [ ]:
N=101; x=np.linspace(0,L,N); dx=x[1]-x[0]
A=np.zeros((N,N)); b=np.zeros(N)
for i in range(1,N-1):
    A[i,i-1]=1; A[i,i]=-2; A[i,i+1]=1; b[i]=-g/k*dx**2
A[0,0]=-3; A[0,1]=4; A[0,2]=-1; b[0]=0.0                      # insulated back
A[-1,-1]=3*k/(2*dx)+h; A[-1,-2]=-4*k/(2*dx); A[-1,-3]=k/(2*dx); b[-1]=h*Tinf  # convective face
T_fd=np.linalg.solve(A,b)
print(f'FD vs analytic max error = {np.max(np.abs(T_fd-T_analytic(x))):.3e} K  (T0={T_fd[0]:.1f}, TL={T_fd[-1]:.1f})')


### 2. The PINN
We nondimensionalize for stable training: let $\xi=x/L$ and $\hat\theta=(T-T_\infty)/100$, so the network output is $O(1)$.
The loss is the **PDE residual** plus the **two boundary residuals** — no labelled solution data is used.

In these variables: $\hat\theta''(\xi)=-0.1$, $\hat\theta'(0)=0$, and $\hat\theta'(1)=-0.1\,\hat\theta(1)$.


In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,1))
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
xi  = torch.linspace(0,1,101).view(-1,1).requires_grad_(True)   # collocation points

def derivs(f, x):
    df  = torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
    d2f = torch.autograd.grad(df, x, torch.ones_like(df), create_graph=True)[0]
    return df, d2f

for it in range(5000):
    opt.zero_grad()
    th = net(xi)
    dth, d2th = derivs(th, xi)
    res_pde = d2th + 0.1                       # theta_hat'' = -0.1
    bc0 = dth[0]                               # insulated back
    bc1 = dth[-1] + 0.1*th[-1]                 # convective face
    loss = (res_pde**2).mean() + bc0**2 + bc1**2
    loss.backward(); opt.step()
    if it % 1000 == 0: print(f'iter {it:5d}   loss {loss.item():.3e}')


### 3. Recover T(x) and compare to the exact answer


In [ ]:
xis = torch.linspace(0,1,101).view(-1,1)
T_pinn = (Tinf + 100*net(xis).detach().numpy().ravel())
xx = xis.numpy().ravel()*L
print(f'PINN max error vs analytic = {np.max(np.abs(T_pinn - T_analytic(xx))):.3f} K')
plt.figure(figsize=(6,4))
plt.plot(xx, T_analytic(xx), 'k', lw=2, label='analytical  405-2000x^2')
plt.plot(x, T_fd, 'b.', ms=4, label='finite difference')
plt.plot(xx, T_pinn, 'r--', lw=1.6, label='PINN')
plt.xlabel('x (m)'); plt.ylabel('T (K)'); plt.legend(); plt.title('Lab 2.1 — PINN vs. physics')
plt.tight_layout(); plt.show()


### 4. Reflect & extend
- The PINN learned the temperature field from the **equation alone** — no solution data — and matches the analytical curve.
- **Verification habit:** we trusted it only because it reproduced a result we could derive by hand.
- **Lab 2.2 (inverse):** delete the value of $\dot g$, add a few “sensor” temperatures from `T_fd`, and make $\dot g$ a trainable parameter the PINN recovers — the basis of infrared-camera diagnostics.
